In [26]:
import numpy as np
import pandas as pd 

from scipy import stats

In [59]:
path="/home/dexter/Documents/GitHub/3D-textures/assets/experimental_corrected_results.csv"
df = pd.read_csv(path)
path="/home/dexter/Documents/GitHub/3D-textures/assets/experimental_method_results.csv"
dft = pd.read_csv(path)

def remove_outliers(group):
    q1 = group["Raw_Avg_Distance"].quantile(0.25)
    q3 = group["Raw_Avg_Distance"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[
        group["Raw_Avg_Distance"].between(lower, upper)
    ]

q1 = dft.groupby(["Printer", "Texture"])["Raw_Avg_Distance"].transform("quantile", 0.25)
q3 = dft.groupby(["Printer", "Texture"])["Raw_Avg_Distance"].transform("quantile", 0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Keep rows that are NOT outliers
dft = dft[
    dft["Raw_Avg_Distance"].between(lower, upper)
].copy()

q1 = df.groupby(["Printer", "Texture"])["Raw_Avg_Distance"].transform("quantile", 0.25)
q3 = df.groupby(["Printer", "Texture"])["Raw_Avg_Distance"].transform("quantile", 0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Keep rows that are NOT outliers
df_clean = dft[
    df["Raw_Avg_Distance"].between(lower, upper)
].copy()

print(dft.head())
df.head()


  Printer  Texture Test_Type Comparison  Raw_Avg_Distance  Std_Deviation
0       B        5         T     1 vs 2          0.000177       0.000127
1       B        5         T     1 vs 3          0.000158       0.000090
2       B        5         T     1 vs 4          0.000172       0.000110
4       B        5         T     2 vs 1          0.000205       0.000112
5       B        5         T     2 vs 3          0.000183       0.000122


/tmp/ipykernel_5164/4038280665.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_clean = dft[


,Printer,Texture,Test_Type,Comparison,Raw_Avg_Distance,Std_Deviation
0,E,2,T,1 vs 2,0.000242,0.000191
1,E,2,T,1 vs 3,0.000250,0.000204
2,E,2,T,1 vs 4,0.000321,0.000324
3,E,2,T,1 vs 5,0.000357,0.000276
4,E,2,T,2 vs 1,0.000494,0.000449


In [60]:
dft.loc[
            (dft["Printer"] == "E") &
            (dft["Texture"] == 3)
        ].copy()



,Printer,Texture,Test_Type,Comparison,Raw_Avg_Distance,Std_Deviation
160,E,3,T,1 vs 2,0.004263,0.081456
161,E,3,T,1 vs 3,0.001725,0.020348
162,E,3,T,1 vs 4,0.001543,0.019493
163,E,3,T,1 vs 5,0.001720,0.008324
164,E,3,T,2 vs 1,0.001443,0.006067
165,E,3,T,2 vs 3,0.001341,0.007694
166,E,3,T,2 vs 4,0.001295,0.004360
167,E,3,T,2 vs 5,0.001183,0.007285
168,E,3,T,3 vs 1,0.006245,0.097045
170,E,3,T,3 vs 4,0.006421,0.099012


In [61]:
from scipy.stats import ttest_ind
import pandas as pd
pd.set_option('display.float_format', '{:.6f}'.format)
printers = ["E", "B", "R"]

print("Printer\t texture \t raw average \t raw std \t method average \t method std \t p-value \t n")

new_table = {
    "Printer": [],
    "Texture": [],
    "Raw AVG": [],
    "Raw STD": [],
    "Method AVG": [],
    "Method STD": [],
    "N": [],
    "N method": [],
    "T-test p-value": []
}

for j in range(3):
    for i in range(1, 7):

        # Raw data
        raw_result = df.loc[
            (df["Printer"] == printers[j]) &
            (df["Texture"] == i)
        ].copy()

        raw_data = raw_result["Raw_Avg_Distance"].dropna()

        avg_raw = raw_data.mean() * 10000
        std_raw = raw_data.std() * 10000
        n_raw = len(raw_data)

        # Method data
        method_result = dft.loc[
            (dft["Printer"] == printers[j]) &
            (dft["Texture"] == i)
        ].copy()

        method_data = method_result["Raw_Avg_Distance"].dropna()

        avg_method = method_data.mean() * 10000
        std_method = method_data.std() * 10000
        n_method = len(method_data)

        t_stat, p_two_sided = ttest_ind(
            raw_data,
            method_data,
            equal_var=False
        )

        if t_stat > 0:
            p_value = p_two_sided / 2
        else:
            p_value = 1 - (p_two_sided / 2)

        # Store results
        new_table["Printer"].append(printers[j])
        new_table["Texture"].append(i)

        new_table["Raw AVG"].append(avg_raw)
        new_table["Raw STD"].append(std_raw)

        new_table["Method AVG"].append(avg_method)
        new_table["Method STD"].append(std_method)

        new_table["N"].append(n_raw)
        new_table["N method"].append(n_method)

        new_table["T-test p-value"].append(p_value)


new_table = pd.DataFrame(new_table)

new_table

Printer	 texture 	 raw average 	 raw std 	 method average 	 method std 	 p-value 	 n


,Printer,Texture,Raw AVG,Raw STD,Method AVG,Method STD,N,N method,T-test p-value
0,E,1,4.024910,1.318906,1.873632,0.305124,20,19,0.000000
1,E,2,3.628370,1.074559,2.731070,1.403772,20,20,0.014687
2,E,3,3.313160,0.521236,38.697767,26.626536,20,18,0.999985
3,E,4,4.103940,0.951012,1.804530,0.346053,20,20,0.000000
4,E,5,4.424970,1.232896,1.949174,0.330192,20,19,0.000000
5,E,6,6.195480,2.338720,2.084744,0.378344,20,18,0.000000
6,B,1,2.552415,0.386546,1.899380,0.317494,20,20,0.000001
7,B,2,2.365715,0.164711,1.582861,0.201939,20,18,0.000000
8,B,3,2.916625,0.782889,1.690484,0.204358,20,19,0.000000
9,B,4,2.282430,0.389422,1.560505,0.292151,20,19,0.000000
